## Setting Up Ollama on Google Colab and Running Gemma4-12B

### Installation

In [ ]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (16.2 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently 

In [ ]:
#WARNING: Unable to detect NVIDIA/AMD GPU. Install lspci or lshw to automatically detect and install GPU dependencies.
!sudo apt update && sudo apt install pciutils lshw

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd

In [ ]:
!pip install openai

### GPU Enabled

In [ ]:
!nvidia-smi

Sat Jun  6 08:23:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Running Ollama Server

Running on a cell leads to getting stuck in that cell, so run it as background process.

In [ ]:
# !ollama serve

In [ ]:
# 1. Kill any existing instances
# !pkill ollama

# 2. Set the environment variable to allow connections
import os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# 3. Start the server in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# 4. Wait for it to warm up
time.sleep(10)
print("Server should be ready.")

Server should be ready.


### Testing if Ollama is Running

In [ ]:
!curl http://localhost:11434

Ollama is running

### Install Ollama Python SDK

It makes using Ollama easy. We can access Ollama programmatically.

In [ ]:
!pip install ollama

### Download Gemma4:12b Model: Run this on terminal

`ollama pull gemma4:12b`

### Checking Model List In Ollama

In [ ]:
!ollama list

NAME    ID    SIZE    MODIFIED 


### Testing Gemma4 on Text Input

In [ ]:
import ollama
response = ollama.chat(model='gemma4:12b', messages=[
  {'role': 'user', 'content': 'Why is the sky blue?'},
])
print(response['message']['content'])


The short answer is that the sky is blue because of a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white to us, it actually contains all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength. Red light has long, lazy waves, while blue and violet light have much shorter, choppier waves.

### 2. The Earth’s atmosphere is like an obstacle course
Our atmosphere is filled with gases (mostly nitrogen and oxygen) and other particles. When sunlight hits these gas molecules, the light is scattered in different directions.

### 3. Shorter waves scatter more easily
Because blue light travels in shorter, smaller waves, it strikes the molecules in the air and gets scattered much more easily than red or yellow light. Because this blue light is bouncing around in every direction as it travels th

### Download an Image

In [ ]:
# !wget https://upload.wikimedia.org/wikipedia/commons/d/d9/Collage_of_Nine_Dogs.jpg

### Testing Gemma4 On Image Input

In [ ]:
import base64

# --- How to prepare your image ---
with open("./Collage_of_Nine_Dogs.jpg", "rb") as image_file:
  encoded_image = base64.b64encode(image_file.read()).decode('utf-8')


# Make the call to the model with the image
response = ollama.chat(
    model='gemma4:12b',
    messages=[
        {
            'role': 'user',
            'content': 'Describe this image.', # content should be a string
            'images': [encoded_image] # images is a list of base64 strings within the message
        }
    ]
)

print(response['message']['content'])

This image is a collage featuring nine different photographs of various breeds of dogs, arranged in a 3x3 grid. Each photo showcases a different breed with unique coat textures, colors, and sizes:

*   **Top Left:** A close-up portrait of a Golden Retriever with golden fur resting its head on green grass.
*   **Top Middle:** A side profile of a long-haired dog with wavy, reddish-gold fur standing in front of a wooden fence. 
*   **Top Right:** A fluffy, curly-haired dog with cream-colored fur, possibly a Goldendoodle or similar poodle cross.
*   **Middle Left:** A long-haired dog with a thick coat and prominent facial hair, showing colors of gray, brown, and black.
*   **Middle Center:** A large, muscular short-haired dog with fawn-colored fur and white markings on its chest. 
*   **Middle Right:** A small, fluffy dog with reddish-brown fur, similar in appearance to a Pomeranian.
*   **Bottom Left:** A tricolor Beagle-type dog lying on the ground with its tongue sticking out.
*   **Bot

In [ ]:
import base64
import ollama
import librosa

def prepare_audio_for_gemma(audio_file_path):
    # Load and resample to 16 kHz mono as required by Gemma 4
    y, sr = librosa.load(audio_file_path, sr=16000, mono=True)
    # Scale to [-1, 1] as float32
    audio_bytes = y.tobytes()
    return base64.b64encode(audio_bytes).decode('utf-8')

# 1. Prepare your audio file
audio_path = "./sample_audio.wav"
encoded_audio = prepare_audio_for_gemma(audio_path)

# 2. Define the message payload
messages = [
    {
        'role': 'user',
        'content': 'Transcribe the following audio segment in English into text:',
    },
    {
        'role': 'user',
        # Attach the audio here. You can pass multiple audio chunks or interleave with text.
        'content': encoded_audio,
    }
]

# 3. Call the Gemma 4 12B model via Ollama
response = ollama.chat(
    model='gemma4:12b',
    messages=messages
)

print(response['message']['content'])

In [ ]:
print(response)

model='gemma4:12b' created_at='2026-06-06T09:13:00.316600732Z' done=True done_reason='length' total_duration=42673305657 load_duration=27844022000 prompt_eval_count=4095 prompt_eval_duration=6465105000 eval_count=1 eval_duration=1000 message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [ ]:
!nvidia-smi

Fri Jun  5 09:54:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----